# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdsvr/flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Ranking / scoring.** My Lane 4 question is "which visible pages under-capture clicks for their position, and which should an editor review first?" — that is a *"which ones first?"* question, which maps to ranking: the output is a priority score, and quality is measured at the top of the list (precision@K).

It is **not** classification — I don't need a correct yes/no on all 30,000 pages, only a trustworthy *ordering* of the top ~50 the editor will actually see. It is not clustering either: there's a specific decision to support, not an open "what kinds exist?" question. The code below shows why the decision is an ordering problem: the qualifying pool is ~200x bigger than review capacity.

In [6]:
# The decision shape: a pool far bigger than capacity -> ordering, not labeling.
import os
import pandas as pd
import numpy as np

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    os.chdir("../..")  # work/notebooks/ -> repo root

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Lane 4 slice: visible pages with position data (gotcha: avg_position == 0 means "no data")
lane = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)]

REVIEW_CAPACITY = 50
print(f"Pages in the lane slice:  {len(lane):,}")
print(f"Editor capacity:          {REVIEW_CAPACITY} per cycle")
print(f"So the task is: order {len(lane):,} candidates so the best {REVIEW_CAPACITY} surface first.")
print("A classifier's yes/no on every page would still leave the ordering question unanswered.")

Pages in the lane slice:  12,023
Editor capacity:          50 per cycle
So the task is: order 12,023 candidates so the best 50 surface first.
A classifier's yes/no on every page would still leave the ordering question unanswered.


## 2. Target or proxy

**Proxy now, observed target later — and I want to be precise about which is which.**

- **What I'd ultimately predict (observed target, Week 3+ on the warehouse):** a future-window outcome — *given features from a prior 90-day window, does the page's CTR/clicks improve or decline over the NEXT 30 days?* That label is **observed** (measured later in time), which is what the framing rule demands: a target the world produces, not one I define.
- **What the starter CSV supports today (proxy):** a **position-adjusted CTR gap score** — each page's CTR versus the median CTR of visible peers in the same position tier. This is a *score I compute*, not a label I train on. If I invented a threshold ("gap < X = underperformer") and trained a model on it, the model would just learn my threshold back — the leakage lesson from notebook 02.
- The starter's own `is_declining_label` (`trend_direction == "down"`) is a current-window derived bucket — usable as a beginner proxy for ranking evaluation, and I use it that way in section 3, but it is not the capstone target.

The code below sketches what the proxy column looks like on real rows.

In [7]:
# Sketch of the proxy: position-adjusted CTR gap (a SCORE, not a training label).
# ctr is a x100 percentage (0.23 means 0.23%) per docs/data-dictionary.md.
visible = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)]
expected = visible.groupby("position_tier")["ctr"].median().rename("expected_ctr_tier")

sketch = lane.join(expected, on="position_tier")
sketch["ctr_gap"] = (sketch["ctr"] - sketch["expected_ctr_tier"]).round(3)

cols = ["content_id", "position_tier", "impressions_90d", "ctr", "expected_ctr_tier", "ctr_gap"]
print(sketch[cols].sort_values("ctr_gap").head(8).to_string(index=False))
print()
print("Most-negative ctr_gap = pages capturing far fewer clicks than peers at the SAME position.")
print("Future observed target (warehouse, Week 3+): next-30-day CTR/clicks change after a")
print("prior-90-day feature window — measured later in time, so it cannot leak into features.")

          content_id position_tier  impressions_90d  ctr  expected_ctr_tier  ctr_gap
content_d10d476fbfa5        page_1             1113  0.0               0.23    -0.23
content_b23fa9e12c1d        page_1             1133  0.0               0.23    -0.23
content_531e5e5a3c0f        page_1              673  0.0               0.23    -0.23
content_2dbab51b83c9        page_1              781  0.0               0.23    -0.23
content_09b0e4b05a30        page_1             4147  0.0               0.23    -0.23
content_d1268550d1e3        page_1             2126  0.0               0.23    -0.23
content_bb485e06e530        page_1             2080  0.0               0.23    -0.23
content_8f2559c3bc1b        page_1              740  0.0               0.23    -0.23

Most-negative ctr_gap = pages capturing far fewer clicks than peers at the SAME position.
Future observed target (warehouse, Week 3+): next-30-day CTR/clicks change after a
prior-90-day feature window — measured later in time, so it c

## 3. Success metric

**Precision@50** — of the top 50 pages my ranking surfaces, what fraction are genuinely worth an editor's attention (scored against the evaluation label). Fifty because that is the review capacity the queue feeds; a metric averaged over all 30,000 pages would reward being right about pages nobody will ever look at.

**What number means "good"? Two bars, both computed below:**
1. **Beat random:** in my lane slice the declining base rate is ~0.60, so a random top-50 already scores ~0.60. Any precision@50 that doesn't clearly beat that is noise.
2. **Beat the transparent rule:** the hand-written baseline is the real competitor. If a learned ranking can't beat the rule at precision@50 on **held-out clients**, the rule wins and I ship the rule.

Secondary check: average precision (whole-ranking quality), but the defended metric is precision@50.

In [8]:
# The two bars any ranking must clear.
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y = lane["trend_direction"].str.lower().eq("down").astype(int).values  # starter proxy label

base_rate = y.mean()
rng = np.random.RandomState(0)
random_p50 = np.mean([precision_at_k(rng.rand(len(y)), y) for _ in range(200)])

print(f"Bar 1 — random:  base rate in lane slice = {base_rate:.3f}; "
      f"random ordering precision@50 = {random_p50:.3f} (mean of 200 draws)")
print(f"Bar 2 — the rule: the transparent baseline from the starter pipeline, ")
print(f"                  re-evaluated on held-out clients (Week 4+).")
print()
print(f"'Good' = precision@50 clearly above {base_rate:.2f} AND above the rule, on clients")
print(f"the model never trained on. In-sample wins don't count (notebook 02 lesson).")

Bar 1 — random:  base rate in lane slice = 0.599; random ordering precision@50 = 0.589 (mean of 200 draws)
Bar 2 — the rule: the transparent baseline from the starter pipeline, 
                  re-evaluated on held-out clients (Week 4+).

'Good' = precision@50 clearly above 0.60 AND above the rule, on clients
the model never trained on. In-sample wins don't count (notebook 02 lesson).


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item for one client, described by its trailing-90-day search behavior.** Not a page-day, not a client: the editor's decision ("review this page or not") happens at the content-item level, so the model's rows must live there too.

The dataframe below is my lane's actual slice: visible pages (≥500 impressions in 90 days) holding a real position (1–20). The grain check confirms one row per content item. On the warehouse release (Week 3+) I'll rebuild this same grain by aggregating `fact_content_daily_performance` per content item — the unit of analysis stays the same, only the source gets richer.

In [9]:
# The lane slice as a real dataframe: one row = one content item.
cols = ["content_id", "client_id", "content_type", "position_tier",
        "impressions_90d", "clicks_90d", "ctr", "avg_position", "days_since_last_update"]
print(f"shape: {lane.shape[0]:,} rows x {lane.shape[1]} columns | "
      f"clients: {lane['client_id'].nunique()}")
assert lane["content_id"].is_unique, "grain violated: expected one row per content item"
print("grain check passed: content_id is unique (one row = one content item)")
lane[cols].head(5)

shape: 12,023 rows x 44 columns | clients: 28
grain check passed: content_id is unique (one row = one content item)


,content_id,client_id,content_type,position_tier,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update
0,content_304f48230142,client_f369cb89fc,keyword article,striking,3803,29,0.76,10.6,20
3,content_331d6c4de07b,client_19581e27de,keyword article,page_1,11751,58,0.49,6.2,22
5,content_d4084a4bc775,client_f369cb89fc,keyword article,page_1,3970,1,0.03,8.5,20
9,content_c27558df2b0c,client_19581e27de,keyword article,page_1,1240,2,0.16,4.9,104
10,content_d8ee6cc6d642,client_19581e27de,keyword article,top_3,20919,324,1.55,2.2,104


## 5. Why ML beats a fixed rule here

Because **the same CTR number means opposite things depending on where the page sits** — and the norms shift again by content type and volume. The code below shows it: a 0.10% CTR is *above* the median for a page ranking on pages 3–5, but less than half the norm for a page-1 page. A fixed rule (`ctr < 0.5`) is blind to that: it flags 9,759 pages — 195x capacity — and cannot say which 50 first.

To hand-write position-and-type-aware thresholds I'd need a rule per (position tier × content type × volume bucket) cell — dozens of thresholds, each drifting as clients and inventory change. That's exactly the "real but too messy to write by hand" pattern where a learned ranking earns its place — **conditionally**: it must beat the transparent rule at precision@50 on held-out clients, or the rule ships.

Careful words: everything here is observed/directional in this pseudonymized dataset — decision support for a human editor, not causal claims about what a rewrite will do, and not claims about Google's algorithm.

In [10]:
# Same number, opposite meanings: why one global threshold can't work.
med = visible.groupby("position_tier")["ctr"].median().round(3)
print("Median CTR (%) of visible pages by position tier:")
print(med.to_string())
print()
example = 0.10
for tier in ["page_3_5", "page_1"]:
    verdict = "ABOVE the tier norm (fine)" if example > med[tier] else "well BELOW the tier norm (review candidate)"
    print(f"A {example:.2f}% CTR at {tier:<9} is {verdict}")
print()
n_cells = visible.groupby(["position_tier", "content_type"]).ngroups
print(f"Hand-writing norms would need thresholds for {n_cells} position x content-type cells")
print(f"(more once volume buckets are added) — each one drifting as inventory changes.")

Median CTR (%) of visible pages by position tier:
position_tier
deep        0.00
page_1      0.23
page_3_5    0.06
striking    0.15
top_3       0.19

A 0.10% CTR at page_3_5  is ABOVE the tier norm (fine)
A 0.10% CTR at page_1    is well BELOW the tier norm (review candidate)

Hand-writing norms would need thresholds for 14 position x content-type cells
(more once volume buckets are added) — each one drifting as inventory changes.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.